In [ ]:
from aux import *
import matplotlib.pyplot as plt

df = pd.read_pickle("data_family.pkl")
exog = df['exog'].copy().ffill()
sales_by_family = df['sales_by_family'] 
familias = pd.read_csv("familias.csv")
data_by_family = {
    fam: format_as_year_month(sales_by_family[sales_by_family['family'] == fam].copy())
    for fam in sales_by_family['family'].unique()
}
#for fam in data_by_family:
#    data_by_family[fam], _ = limpiar_outliers_x_agrupacion(data_by_family[fam], 'family', 'sale_amount_MM')
name = '0105'
target_col = 'sale_amount_MM'
ignore_col = [] 
negatives_reg_col = []
feat = feature_selection(data_by_family[name], exog, max_lag=4, min_lag=-3,
                         target_col=target_col, ignore_col=ignore_col,
                         negatives_reg_col=negatives_reg_col)
feat = feat[feat['correlación'] > 0.4].reset_index(drop=True)
feat = clean_focus_correlation(feat, group='variable', focus='correlación')
feat = feat.sort_values(by='correlación', ascending=False).reset_index(drop=True)
selected, resumen = collinearity_analysis(data_by_family[name], exog, feat, target_col)
feat = feat[feat['variable'].isin(selected)]
df_final = construir_dataset_familia(name, data_by_family, exog, feat, target_col)
splits = generar_splits(df_final)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_family_sales(data_by_family, familias, names, fig_width=18, fig_height=10):
    rows = len(names)
    fig, axes = plt.subplots(rows, 1, figsize=(fig_width, fig_height), sharex=True,
                             constrained_layout=True) 
    if rows == 1:
        axes = [axes]
    for idx, name in enumerate(names):
        serie = data_by_family[name].sale_amount_MM
        x = serie.index
        y = serie.values
        ax = axes[idx]
        ax.plot(x, y, color='steelblue', linewidth=2)
        family_name = familias.loc[familias.hier_family_cd == int(name), 'hier_family_name'].values[0]
        ax.set_title(f"Evolución de Ventas Mensuales - {family_name}", fontsize=13)
        ax.set_ylabel("Ventas (MM)", fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.5)
        if x.dtype.kind in {'M', 'm'}:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
            ax.tick_params(axis='x', rotation=45)
    axes[-1].set_xlabel("Fecha", fontsize=12)
    plt.show()
names = [ '0105', '0421','0312']  # lista de códigos de familia que quieres graficar
plot_family_sales(data_by_family, familias, names)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
import torch
_llm_model = None
_tokenizer = None
def fit_predict_eval_llm_forecaster(training_set, test_set, model_params=None):
    """
    Simula la predicción de series de tiempo usando un LLM como si fuera un modelo de forecasting.
    Transforma la serie temporal a texto, el modelo predice tokens que representan valores futuros.
    """
    global _llm_model, _tokenizer

    default_params = {
        'context_length': 128,
        'prediction_length': 24,
        'model_name': 'gpt2'  # o uno de huggingface como mistralai/Mistral-7B-Instruct-v0.2
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
    y_test = scaler_y.transform(test_set[['y']]).flatten()
    y_all = np.concatenate([y_train, y_test])
    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

    # Inicializa el modelo y tokenizer
    if _llm_model is None or _tokenizer is None:
        _tokenizer = AutoTokenizer.from_pretrained(p['model_name'])
        _llm_model = AutoModelForCausalLM.from_pretrained(p['model_name'])

    # Codifica la serie como texto (e.g., coma separada)
    context_series = y_all[-total_required_length:-p['prediction_length']]
    context_str = ", ".join([f"{v:.3f}" for v in context_series])
    prompt = f"Given the previous values: [{context_str}], predict the next {p['prediction_length']} values:"

    inputs = _tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = _llm_model.generate(**inputs, max_new_tokens=50, pad_token_id=_tokenizer.eos_token_id)

    decoded = _tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extraer los números predichos desde el texto (simple parsing, no robusto)
    import re
    predicted_str = decoded.split("predict the next")[1]
    predicted_values = re.findall(r"\d+\.\d+", predicted_str)
    y_pred_scaled = np.array(predicted_values[:p['prediction_length']], dtype=float)

    # Inversa del escalamiento
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    index_pred = test_set.index[:len(y_pred)]
    return _llm_model, pd.Series(y_pred, index=index_pred, name='LLMForecast'), scaler_y

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_llm_forecaster_cv(df, split, n_trials=10):
    """
    Optimiza la longitud de contexto y predicción de un modelo LLM para series de tiempo usando validación cruzada.
    """
    model_dict = {}

    def objective(trial):
        model_params = {
            'context_length': trial.suggest_int('context_length', 24, 72),  # 2 a 6 años (mensual)
            'prediction_length': trial.suggest_int('prediction_length', 6, 18),  # 6 a 18 meses
            'model_name': 'gpt2'  # puedes probar otros como 'tiiuae/falcon-rw-1b' o 'mistralai/Mistral-7B-Instruct-v0.2'
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                total_len = len(training_set) + len(test_set)
                required_len = model_params['context_length'] + model_params['prediction_length']

                if total_len < required_len:
                    raise ValueError("No hay suficientes datos para el contexto y predicción requeridos por el modelo.")

                _, y_pred, _ = fit_predict_eval_llm_forecaster(training_set, test_set, model_params)
                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)

                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] LLM Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from llama_cpp import Llama
import numpy as np
import pandas as pd
import re
from llama_cpp import Llama
_llm_gguf_model = None  # Global para evitar recarga

def fit_predict_eval_llm_forecaster_gguf(training_set, test_set, model_params=None):
    """
    Simula la predicción de series de tiempo usando un LLM en formato .gguf.
    Si normalize=True, aplica MinMaxScaler. Si no, usa los valores reales.
    """
    global _llm_gguf_model

    default_params = {
        'context_length': 128,
        'prediction_length': 24,
        'model_path': './models/llama-2-7b.Q4_K_M.gguf',
        'max_tokens': 256,
        'n_ctx': 2048,
        'normalize': False
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    # Validar tamaño
    if p['prediction_length'] > len(test_set):
        raise ValueError(f"prediction_length ({p['prediction_length']}) > test_set ({len(test_set)}).")

    # Normalizar si se indica
    if p['normalize']:
        scaler_y = MinMaxScaler()
        if training_set[['y']].dropna().empty:
            raise ValueError("training_set['y'] está vacío o solo tiene NaN")

        y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
        y_test = scaler_y.transform(test_set[['y']]).flatten()
        y_all = np.concatenate([y_train, y_test])
    else:
        scaler_y = None
        y_train = training_set['y'].values
        y_test = test_set['y'].values
        y_all = np.concatenate([y_train, y_test])

    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para contexto + predicción.")

    # Cargar modelo solo una vez
    if _llm_gguf_model is None:
        _llm_gguf_model = Llama(model_path=p['model_path'], n_ctx=p['n_ctx'])



    context_series = y_all[-total_required_length:-p['prediction_length']]
    context_str = ", ".join([f"{v:.3f}" for v in context_series])
    prompt = f"Given the previous values: [{context_str}], predict the next {p['prediction_length']} values:"    

    # Crear prompt con ejemplo opcional
    #context_series = y_all[-total_required_length:-p['prediction_length']]
    #context_str = ", ".join([f"{v:.3f}" for v in context_series])
    #prompt = (
    #    f"Example:\n"
    #    f"Input: 0.2, 0.3, 0.4\n"
    #    f"Output: 0.5, 0.6, 0.7\n\n"
    #    f"Input: {context_str}\n"
    #    f"Output:"
   # )

    output = _llm_gguf_model(prompt, max_tokens=p['max_tokens'], echo=False)
    generated_text = output['choices'][0]['text'].strip()

    # Extraer números
    predicted_values = re.findall(r"\d+\.\d+", generated_text)
    if not predicted_values:
        raise ValueError("El modelo no generó ningún número reconocible.")

    y_pred_raw = np.array(predicted_values, dtype=float)

    # Limitar al tamaño de test_set
    n_values = min(len(y_pred_raw), len(test_set), p['prediction_length'])

    if scaler_y:
        y_pred = scaler_y.inverse_transform(y_pred_raw[:n_values].reshape(-1, 1)).flatten()
    else:
        y_pred = y_pred_raw[:n_values]

    index_pred = test_set.index[:n_values]

    return _llm_gguf_model, pd.Series(y_pred, index=index_pred, name='LLMForecast_GGUF'), scaler_y

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_llm_forecaster_cv_gguf(df, split, n_trials=10):
    """
    Optimiza hiperparámetros para el LLM forecaster en .gguf con validación cruzada y normalización opcional.
    """
    model_dict = {}

    def objective(trial):
        # Ajuste dinámico al tamaño mínimo de test_set
        min_test_len = min(len(df.iloc[test_index]) for _, test_index in split)

        model_params = {
            'context_length': trial.suggest_int('context_length', 24, 72),
            'prediction_length': trial.suggest_int('prediction_length', 9, min(18, min_test_len)),
            'model_path': './models/llama-2-7b.Q4_K_M.gguf',
            #'max_tokens': trial.suggest_int('max_tokens', 128, 512),
            'n_ctx':1054 ,
            'normalize': trial.suggest_categorical('normalize', [True, False])
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                total_len = len(training_set) + len(test_set)
                required_len = model_params['context_length'] + model_params['prediction_length']

                if total_len < required_len:
                    raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

                _, y_pred, _ = fit_predict_eval_llm_forecaster_gguf(training_set, test_set, model_params)

                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)

                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

class LSTMExogenous(nn.Module):
    def __init__(self, input_size_seq, input_size_exog, hidden_size=64,
                 num_layers=1, dropout=0.2, prediction_length=6):
        super().__init__()
        self.lstm = nn.LSTM(input_size_seq, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc_exog = nn.Linear(input_size_exog, hidden_size)
        self.fc_out = nn.Linear(hidden_size * 2, prediction_length)

    def forward(self, x_seq, x_exog):
        lstm_out, _ = self.lstm(x_seq)
        lstm_last = lstm_out[:, -1, :]
        exog_out = torch.relu(self.fc_exog(x_exog))
        combined = torch.cat([lstm_last, exog_out], dim=1)
        output = self.fc_out(combined)
        return output

def fit_predict_eval_lstm(training_set, test_set, model_params=None,
                          seq_len=12, prediction_length=6,
                          epochs=50, lr=0.001, batch_size=32):
    default_params = dict(
        input_size_seq=1,
        input_size_exog=len(training_set.columns.difference(['ds', 'y'])),
        hidden_size=64,
        num_layers=1,
        dropout=0.2
    )
    if model_params:
        default_params.update(model_params)

    scaler_y = StandardScaler()
    scaler_exog = StandardScaler()

    def build_sequences(df):
        y_scaled = scaler_y.fit_transform(df[['y']])
        exog_scaled = scaler_exog.fit_transform(df.drop(columns=['ds', 'y']))

        seqs, exogs, targets = [], [], []
        for i in range(len(df) - seq_len - prediction_length + 1):
            seqs.append(y_scaled[i:i+seq_len])
            exogs.append(exog_scaled[i+seq_len])
            targets.append(y_scaled[i+seq_len:i+seq_len+prediction_length].flatten())

        return (torch.tensor(seqs, dtype=torch.float32),
                torch.tensor(exogs, dtype=torch.float32),
                torch.tensor(targets, dtype=torch.float32))

    x_seq_train, x_exog_train, y_train = build_sequences(training_set)
    x_seq_test, x_exog_test, y_test = build_sequences(
        pd.concat([training_set.tail(seq_len + prediction_length - 1), test_set])
    )

    train_ds = TensorDataset(x_seq_train, x_exog_train, y_train)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = LSTMExogenous(**default_params, prediction_length=prediction_length)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        for xb_seq, xb_exog, yb in train_dl:
            optimizer.zero_grad()
            preds = model(xb_seq, xb_exog)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(x_seq_test, x_exog_test).numpy()
        preds_inv = scaler_y.inverse_transform(preds)

    # Corregir el index de predicciones
    pred_index = test_set['ds'].reset_index(drop=True)
    pred_dates = pred_index.iloc[:len(preds_inv)]  # evita KeyError

    columns = [f"step_{i+1}" for i in range(prediction_length)]
    predictions = pd.DataFrame(preds_inv, index=pred_dates, columns=columns)

    return model, predictions, None

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_lstm_cv(df, split, n_trials=10):
    model_dict = {}

    def objective(trial):
        model_params = {
            'hidden_size': trial.suggest_int('hidden_size', 32, 128),
            'num_layers': trial.suggest_int('num_layers', 1, 4),
            'dropout': trial.suggest_float('dropout', 0.0, 0.5),
        }

        # Ajuste por warning de PyTorch si num_layers = 1
        if model_params['num_layers'] == 1:
            model_params['dropout'] = 0.0

        fit_params = {
            'seq_len': trial.suggest_int('seq_len', 6, 24),
            'epochs': trial.suggest_int('epochs', 20, 100),
            'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32])
        }

        prediction_length = 6  # fijo o puedes tunear también
        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                _, y_pred_df, _ = fit_predict_eval_lstm(
                    training_set,
                    test_set,
                    model_params=model_params,
                    seq_len=fit_params['seq_len'],
                    epochs=fit_params['epochs'],
                    lr=fit_params['lr'],
                    batch_size=fit_params['batch_size']
                )

                if y_pred_df is None or y_pred_df.empty:
                    raise ValueError("Predicción inválida: resultado vacío.")

                # Alineamos con los datos reales del test
                n_preds = len(y_pred_df)
                y_true = test_set['y'].iloc[:n_preds].values
                y_pred = y_pred_df.iloc[:, 0].values  # usamos solo el primer paso de predicción

                if len(y_true) != len(y_pred):
                    raise ValueError("Predicción inválida: longitud de alineación insuficiente.")

                mape = mean_absolute_percentage_error(y_true, y_pred)
                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': {**model_params, **fit_params},
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            import traceback
            print(f"[ERROR] Trial {trial.number}  | {type(e).__name__}: {e}")
            traceback.print_exc()
            return float("inf")

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best'])
    return study.best_params, trials_df, model_dict

In [ ]:
%%capture
from optimizers import optimize_lstm_cv
from optimizers import optimize_gru_cv
from optimizers import optimize_hw_cv
from optimizers import optimize_silverkite_cv
from optimizers import optimize_prophet_cv
from optimizers import optimize_mlp_cv
n_trials=50
#best_params_lstm, trials_df_lstm, model_dict_lstm = optimize_lstm_cv(df_final.assign(ds=df_final.index), splits, n_trials)
best_params_gpt2, trials_df_gpt2, model_dict_gpt2 = optimize_llm_forecaster_cv(df_final.sort_values('ds').reset_index(drop=True), splits, n_trials=50)
#trials_df_lstm['model'] = 'lstm'
trials_df_gpt2['model'] = 'gpt2'
#best_params_hw, trials_df_hw, model_dict_hw = optimize_hw_cv(df_final, splits, n_trials)
#rials_df_hw['model'] = 'holt_winters'
#best_params_sk, trials_df_sk, model_dict_sk = optimize_silverkite_cv(df_final, splits, n_trials)
#trials_df_sk['model'] = 'silverkite'
#best_params_gru, trials_df_gru, model_dict_gru = optimize_gru_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_gru['model'] = 'gru'
#best_params_prophet, trials_df_prophet, model_dict_prophet = optimize_prophet_cv(df_final.reset_index(), splits,n_trials)
#trials_df_prophet['model'] = 'prophet'
#best_params_mlp, trials_df_mlp, model_dict_mlp = optimize_mlp_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_mlp['model'] = 'MLP'
#best_params_gguf, trials_df_gguf, model_dict_gguf = optimize_llm_forecaster_cv_gguf(
    #df_final.sort_values('ds').reset_index(drop=True),
  #  splits,
  #  n_trials=25)
#trials_df_gguf['model'] = 'OLLAMA-GGUF'
#best_params_EXlstm, trials_df_EXlstm, model_dict_EXlstm = optimize_lstm_cv(df_final.assign(ds=df_final.index), splits, n_trials=100)
#trials_df_gguf['model'] = 'EXOG-LSTM'
import pandas as pd
import pickle
results = {}
results['trials'] = pd.concat([
    trials_df_lstm.assign(model='LSTM'),
    trials_df_gpt2.assign(model='GPT2'),
    trials_df_hw.assign(model='HW'),
    trials_df_prophet.assign(model='Prophet'),
    trials_df_mlp.assign(model='MLP'),
    trials_df_lstm.assign(model='LSTM'),
    trials_df_gguf.assign(model='OLLAMA-GGUF'),
    trials_df_EXlstm.assign(model='EXOG-LSTM')
], ignore_index=True)

results['features']=feat
with open('results_new'+name+'.pkl', 'wb') as file:
    pickle.dump(results, file)